In [ ]:
# INSTALL LIBRERIAS
!pip install feedparser
!pip install newspaper3k
!pip install lxml_html_clean

In [8]:
# =========================================
# LIBRERÍAS
# =========================================
import feedparser
from newspaper import Article
import hashlib
import json
import os
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from google.colab import drive


# =========================================
# DRIVE
# =========================================
def conectar_drive():
    drive.mount('/content/drive')


def obtener_file_id(url_drive):
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url_drive)
    if match:
        return match.group(1)
    else:
        raise ValueError("❌ URL de Drive inválida")


def obtener_path_drive(file_id):
    return f"/content/drive/MyDrive/{file_id}.json"


# =========================================
# UTILIDADES
# =========================================
def generar_id(url):
    return hashlib.md5(url.encode()).hexdigest()


def limpiar_texto(texto):
    if not texto:
        return ""
    return re.sub(r"\s+", " ", texto).strip()


def obtener_imagen_og(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        html = requests.get(url, headers=headers, timeout=10).text
        soup = BeautifulSoup(html, "html.parser")

        og = soup.find("meta", property="og:image")
        if og and og.get("content"):
            return og["content"]

    except:
        pass

    return None


# =========================================
# EXTRACCIÓN PRINCIPAL (NEWSPAPER)
# =========================================
def extraer_contenido(url):

    try:
        article = Article(url)
        article.download()
        article.parse()

        titulo = limpiar_texto(article.title)
        texto = limpiar_texto(article.text)
        imagen = article.top_image

        # fallback imagen
        if not imagen or "resizer" in imagen:
            imagen = obtener_imagen_og(url)

        if not imagen:
            print("⚠️ sin imagen válida")

        # fallback contenido mínimo
        if len(texto) < 200:
            return None

        return {
            "titulo": titulo,
            "contenido": texto,
            "imagen": imagen
        }

    except Exception as e:
        print(f"⚠️ Error newspaper: {e}")
        return None


# =========================================
# FORMATEO HTML WORDPRESS
# =========================================
def formatear_html(contenido, link):

    parrafos = contenido.split(". ")

    html = ""

    for p in parrafos:
        if len(p.strip()) > 40:
            html += f"<p>{p.strip()}.</p>"

    html += f'<p><a href="{link}" target="_blank">Fuente original</a></p>'

    return html


# =========================================
# PROCESAR ARTÍCULO
# =========================================
def procesar_entry(entry, categoria):

    url = entry.link

    data = extraer_contenido(url)

    if not data:
        print(f"❌ No se pudo extraer: {url}")
        return None

    contenido_html = formatear_html(data["contenido"], url)

    return {
        "id": generar_id(url),
        "categoria": categoria,
        "titulo": data["titulo"],
        "contenido": contenido_html,
        "imagen_url": data["imagen"],
        "estado": "publish"
    }


# =========================================
# GENERAR POSTS DESDE FEED
# =========================================
def generar_posts(url_feed, categoria, max_items=10):

    feed = feedparser.parse(url_feed)

    if not feed.entries:
        print("❌ Feed vacío")
        return []

    posts = []

    for entry in feed.entries[:max_items]:
        try:
            post = procesar_entry(entry, categoria)
            if post:
                posts.append(post)
        except Exception as e:
            print(f"⚠️ Error: {e}")

    return posts


# =========================================
# GUARDAR EN DRIVE
# =========================================
def guardar_en_drive(posts, path):

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            existentes = json.load(f)
    else:
        existentes = []

    combinados = posts + existentes

    unique = {p["id"]: p for p in combinados}.values()

    with open(path, "w", encoding="utf-8") as f:
        json.dump(list(unique), f, ensure_ascii=False, indent=2)

    print(f"✅ Guardado en: {path}")
    print(f"📰 Total posts: {len(unique)}")

    return list(unique)


# =========================================
# PIPELINE PRINCIPAL (ESTE ES EL QUE TE FALTABA)
# =========================================
def pipeline_news(url_feed, categoria, url_drive, max_items=10):

    print("🚀 Iniciando pipeline...")

    file_id = obtener_file_id(url_drive)
    path = obtener_path_drive(file_id)

    posts = generar_posts(url_feed, categoria, max_items)

    resultado = guardar_en_drive(posts, path)

    print("🏁 Pipeline finalizado")

    return resultado

In [ ]:
conectar_drive()


# EL ID CAMBIA SEGUN EL ARCHIVO JSON

# REEMPLAZAR aaaaaaaaaaaaaaaAaaAaaAaa por el ID
url_drive = "https://drive.google.com/file/d/aaaaaaaaaaaaaaaAaaAaaAaa/view?usp=sharing"


### URL A USAR

url_feed = "https://www.lanacion.com.ar/arc/outboundfeeds/rss/"
categoria = "NOTICIAS"

pipeline_news(url_feed, categoria, url_drive, max_items=5)

url_feed = "https://www.ole.com.ar/rss/ultimas-noticias/"
categoria = "DEPORTES"


pipeline_news(url_feed, categoria, url_drive, max_items=5)